## 第一步：环境配置与路径挂载
> 将项目根目录添加到系统路径，以便直接引用 src 下的模块。

In [4]:
import os
import sys
from torchsummary import summary

# 动态定位项目根目录
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 直接从源文件导入模型类
from src.pneumonia_task.model import PneumoniaCNN, DenoisingAutoencoder

print(f"成功挂载项目路径: {project_root}")

成功挂载项目路径: /home/alpha/ML/code/class/test6


## 第二步：展示 CNN 内部结构与尺寸演变
> 实例化 PneumoniaCNN，并使用 torchsummary 打印出每一层的输出尺寸。

In [5]:
# 1. 初始化模型
img_size = 256
cnn_model = PneumoniaCNN(img_size=img_size)

# 2. 展示结构
print("="*40)
print(f"输入图像尺寸: (1, {img_size}, {img_size})")
print("="*40)

summary(cnn_model, input_size=(1, img_size, img_size), device="cpu")

输入图像尺寸: (1, 256, 256)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 256, 256]             160
              ReLU-2         [-1, 16, 256, 256]               0
         MaxPool2d-3         [-1, 16, 128, 128]               0
            Conv2d-4         [-1, 32, 128, 128]           4,640
              ReLU-5         [-1, 32, 128, 128]               0
         MaxPool2d-6           [-1, 32, 64, 64]               0
            Linear-7                  [-1, 128]      16,777,344
              ReLU-8                  [-1, 128]               0
           Dropout-9                  [-1, 128]               0
           Linear-10                    [-1, 3]             387
Total params: 16,782,531
Trainable params: 16,782,531
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.25
Forward/backward pass size (MB): 27.00
Param

> 网络结构

![](../assets/CNN.png)

## 第三步：展示 DAE 去噪自编码器的尺寸变化
> 展示DAE如何将图像还原回原始尺寸。

In [6]:
# 实例化自编码器
ae_model = DenoisingAutoencoder()

print("="*40)
print("DAE (去噪自编码器) 尺寸演变图")
print("="*40)

# 观察 Encoder 是如何下采样，以及 Decoder 是如何通过 Upsampling 恢复 224x224 的
summary(ae_model, input_size=(1, img_size, img_size), device="cpu")

DAE (去噪自编码器) 尺寸演变图
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 256, 256]             320
              ReLU-2         [-1, 32, 256, 256]               0
         MaxPool2d-3         [-1, 32, 128, 128]               0
            Conv2d-4         [-1, 64, 128, 128]          18,496
              ReLU-5         [-1, 64, 128, 128]               0
         MaxPool2d-6           [-1, 64, 64, 64]               0
            Conv2d-7           [-1, 32, 64, 64]          18,464
              ReLU-8           [-1, 32, 64, 64]               0
UpsamplingNearest2d-9         [-1, 32, 128, 128]               0
           Conv2d-10         [-1, 32, 128, 128]           9,248
             ReLU-11         [-1, 32, 128, 128]               0
UpsamplingNearest2d-12         [-1, 32, 256, 256]               0
           Conv2d-13          [-1, 1, 256, 256]             289
          Sigmoid

![](../assets/DAE.png)